# Chapter 6: Quantization for Small Hardware

*Small Language Models in Practice — Haji Gul*

> Why quantization is the single biggest lever for running models on modest
hardware; what 8-bit and 4-bit actually mean; loading a model in 4-bit with
`bitsandbytes`; and exporting to GGUF for `llama.cpp` and Ollama.

---

*Lecture notes mirroring the book. Run the setup cell, then work top-to-bottom. Swap model ids freely.*

## Setup
Uncomment what this chapter needs.

In [ ]:
# %pip install -q transformers datasets accelerate torch
# Chapter-specific installs appear in shell cells below.

; and exporting to GGUF for `llama.cpp` and Ollama.

## The memory problem

Model weights are numbers. By default each is a 16-bit float, so a 7B-parameter
model needs roughly 7 2 14 GB just to
load — before any activations. That does not fit a typical laptop GPU.
**Quantization** stores each weight in fewer bits, shrinking the model so it
fits, with surprisingly little quality loss.

> **Bits to gigabytes.** Rough memory for a 7B model: **fp16** ≈ 14 GB,
**8-bit** ≈ 7 GB, **4-bit** ≈ 3.5 GB. Four-bit is the
sweet spot for local use: it fits an 8 GB GPU and keeps most of the quality for
chat, RAG, and agent work.

## 4-bit loading with bitsandbytes

This is the most common path: quantize *on load*, directly in
`transformers`. No separate conversion step.

In [ ]:
%%bash
pip install bitsandbytes accelerate

In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
)

quant_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",            # 4-bit NormalFloat, good default
    bnb_4bit_compute_dtype=torch.float16, # math still done in fp16
    bnb_4bit_use_double_quant=True,       # quantize the quantization constants
)

model_id = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id, quantization_config=quant_cfg, device_map="auto",
)

print(model.get_memory_footprint() / 1e9, "GB")   # far below fp16

The model is used exactly like any other — generation code does not change.

In [ ]:
msgs = [{"role": "user", "content": "Name two quantization formats."}]
ids = tokenizer.apply_chat_template(
    msgs, add_generation_prompt=True, return_tensors="pt").to(model.device)
out = model.generate(ids, max_new_tokens=60)
print(tokenizer.decode(out[0][ids.shape[-1]:], skip_special_tokens=True))

> **Tip.** **nf4** (NormalFloat-4) usually beats plain int4 for quality at the same
size, and **double quantization** saves a little more memory for free. These
two defaults are good for almost everything; change them only after measuring.

## GGUF for llama.cpp and Ollama

`bitsandbytes` quantizes for the Python/GPU path. For the
`llama.cpp` ecosystem (CPU-friendly, powers Ollama and LM Studio) the
format is **GGUF**, with quantization levels like `Q4_K_M`.

In [ ]:
%%bash
# Many models are already published as GGUF on the Hub.
# To convert one yourself with llama.cpp tooling:
python convert_hf_to_gguf.py ./my-model --outfile model.f16.gguf
./llama-quantize model.f16.gguf model.Q4_K_M.gguf Q4_K_M

You can then run that GGUF file directly, or import it into Ollama with a
two-line `Modelfile`:

In [ ]:
%%bash
# Modelfile
FROM ./model.Q4_K_M.gguf

# Build and run it
ollama create my-model -f Modelfile
ollama run my-model "Hello from a quantized local model."

> **Which path do I use?.** Training or staying in Python on a GPU **bitsandbytes 4-bit**.
Shipping a single portable file that runs anywhere, including CPU-only laptops
 **GGUF** via `llama.cpp`/Ollama.

## Recap and exercise

You learned what quantization trades and why, loaded a 3B model in 4-bit on
modest hardware, and saw how GGUF serves the CPU-friendly local ecosystem.

**Exercise.** Load the same model in fp16 and in 4-bit, print
`get_memory_footprint()` for each, and compare a few generations.
Decide whether the quality difference is noticeable for your task.